In [4]:
import pandas as pd
import numpy as np
import os
import json
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split

os.makedirs("synthetic_extension", exist_ok=True)

#df_train = pd.read_csv("../datasets/CICEVSE2024/synthetic_extension/train_dataset.csv")
#Use the whole dataset
df_train = pd.read_csv("../datasets/CICEVSE2024/synthetic_extension/EVSE-B-PowerCombined_filtered.csv")

In [5]:
txt = '''
Whether the output should include only the four feature columns plus Attack, or also preserve other columns from the original dataset.
only the four features + attack label

How many synthetic rows you want in total, or per attack class: in total about 34312 rows per attack class around 
none = 29,3%
Backdoor = 43,12%
syn-flood = 27,58 %

reconstructing noisy inputs with a standard denoising autoencoder, without sampling from the latent space

Whether you want the code only, or also saving logic for CSV export.
Also the CSV in the following folder: ../datasets/CICEVSE2024/synthetic_extension/



'''

print(txt)


Whether the output should include only the four feature columns plus Attack, or also preserve other columns from the original dataset.
only the four features + attack label

How many synthetic rows you want in total, or per attack class: in total about 34312 rows per attack class around 
none = 29,3%
Backdoor = 43,12%
syn-flood = 27,58 %

reconstructing noisy inputs with a standard denoising autoencoder, without sampling from the latent space

Whether you want the code only, or also saving logic for CSV export.
Also the CSV in the following folder: ../datasets/CICEVSE2024/synthetic_extension/






In [6]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense, GaussianNoise
from tensorflow.keras.callbacks import EarlyStopping


# --------------------------------------------------
# Reproducibility
# --------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# --------------------------------------------------
# Paths
# --------------------------------------------------
input_path = "../datasets/CICEVSE2024/synthetic_extension/train_dataset.csv"
output_dir = "../datasets/CICEVSE2024/synthetic_extension/"
os.makedirs(output_dir, exist_ok=True)
''

# --------------------------------------------------
# Dataset configuration
# --------------------------------------------------
feature_cols = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]
state_col = "State"
label_col = "Attack"

target_total_rows = 34312
target_distribution = {
    "none": 0.2930,
    "Backdoor": 0.4312,
    "syn-flood": 0.2758
}


# --------------------------------------------------
# Model variants
# baseline_variant: current baseline model
# variant_a: smaller model with smaller latent space
# variant_b: stronger noise for denoising robustness
# variant_c: wider model with larger latent space
# --------------------------------------------------
'''
variants = {  
    "baseline_variant": { #your current balanced starting point, medium noise, medium bottleneck, medium depth
        "corruption_rate": 0.2, # controls denoising difficulty.
        "hidden_layers": [64, 32, 16, 8, 4, 4, 8, 16, 32, 64],
        "latent_dim": 4, # controls compression strength.
        "activation_fn": "relu",
        "epochs": 100,
        "batch_size": 32
    },
    "variant_a": { # a smaller and more compressed model, with a stronger bottleneck effect and less corruption, so it learns a tighter representation
        "corruption_rate": 0.1,
        "hidden_layers": [32, 16, 8, 4, 4, 8, 16, 32],
        "latent_dim": 2,
        "activation_fn": "relu",
        "epochs": 100,
        "batch_size": 32
    },
    "variant_b": { # more robust denoising model, because the corruption rate is higher, so it has to learn to recover from noisier inputs.
        "corruption_rate": 0.3,
        "hidden_layers": [64, 32, 16, 8, 8, 16, 32, 64],
        "latent_dim": 4,
        "activation_fn": "relu",
        "epochs": 100,
        "batch_size": 32
    },
    "variant_c": { # a larger-capacity model, with a wider network and bigger latent dimension, so it can preserve more detail but may be easier to overfit.
        "corruption_rate": 0.2,
        "hidden_layers": [128, 64, 32, 16, 8, 8, 16, 32, 64, 128], # (grid search hyper paprameter tuning, die beste parameter suchen "Optuna")
        "latent_dim": 8,
        "activation_fn": "relu",
        "epochs": 100,
        "batch_size": 32
    }
}
'''
variants = {
    "best_ae_variant": {
        "input_dim": 6,
        "latent_dim": 8,
        "hidden_layers": [256, 128],
        "dropout": 0.0,
        "epochs": 49,
        "batch_size": 64,
        "learning_rate": 5e-05,
        "weight_decay": 0.0
    },

    "variant_a": {
        "input_dim": 6,
        "latent_dim": 8,
        "hidden_layers": [64, 32],
        "dropout": 0.0,
        "epochs": 47,
        "batch_size": 128,
        "learning_rate": 0.001,
        "weight_decay": 0.0
    },

    "variant_b": {
        "input_dim": 6,
        "latent_dim": 8,
        "hidden_layers": [256, 128],
        "dropout": 0.0,
        "epochs": 38,
        "batch_size": 64,
        "learning_rate": 5e-05,
        "weight_decay": 1e-05
    },

    "variant_c": {
        "input_dim": 6,
        "latent_dim": 8,
        "hidden_layers": [256, 128],
        "dropout": 0.0,
        "epochs": 45,
        "batch_size": 128,
        "learning_rate": 1e-04,
        "weight_decay": 0.0
    }
}
#


# --------------------------------------------------
# Load and validate dataset
# --------------------------------------------------
df = pd.read_csv(input_path)

required_cols = feature_cols + [state_col, label_col]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].copy()
df = df[df[label_col].isin(target_distribution.keys())].dropna().reset_index(drop=True)

if df.empty:
    raise ValueError("No valid rows found after filtering for required attack labels.")


# --------------------------------------------------
# Compute target row count per attack label
# --------------------------------------------------
target_counts = {k: int(target_total_rows * v) for k, v in target_distribution.items()}
remainder = target_total_rows - sum(target_counts.values())

if remainder != 0:
    largest_class = max(target_distribution, key=target_distribution.get)
    target_counts[largest_class] += remainder

print("Target counts:", target_counts)


# --------------------------------------------------
# Build denoising autoencoder
# The encoder uses the first half of the hidden layers.
# The decoder uses the second half of the hidden layers.
# --------------------------------------------------
def build_dae(input_dim, hidden_layers, latent_dim, corruption_rate, activation_fn="relu"):
    inp = Input(shape=(input_dim,), name="clean_input")
    x = GaussianNoise(stddev=corruption_rate, name="input_noise")(inp)

    split_index = len(hidden_layers) // 2
    encoder_layers = hidden_layers[:split_index]
    decoder_layers = hidden_layers[split_index:]

    for i, units in enumerate(encoder_layers):
        x = Dense(units, activation=activation_fn, name=f"encoder_dense_{i+1}")(x)

    latent = Dense(latent_dim, activation=activation_fn, name="latent")(x)

    x = latent
    for i, units in enumerate(decoder_layers):
        x = Dense(units, activation=activation_fn, name=f"decoder_dense_{i+1}")(x)

    out = Dense(input_dim, activation="linear", name="reconstruction")(x)

    autoencoder = Model(inputs=inp, outputs=out, name="denoising_autoencoder")
    autoencoder.compile(optimizer="adam", loss="mse")
    return autoencoder


# --------------------------------------------------
# Train and generate synthetic data for one variant
# --------------------------------------------------
def generate_synthetic_dataset(df, variant_name, config):
    corruption_rate = config["corruption_rate"]
    hidden_layers = config["hidden_layers"]
    latent_dim = config["latent_dim"]
    activation_fn = config["activation_fn"]
    epochs = config["epochs"]
    batch_size = config["batch_size"]

    print(f"\n{'=' * 60}")
    print(f"Running variant: {variant_name}")
    print(f"Configuration: {config}")
    print(f"{'=' * 60}")

    early_stopping = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=10,
        min_delta=1e-4,
        restore_best_weights=True,
        start_from_epoch=10,
        verbose=1
    )

    synthetic_parts = []
    metrics_rows = []

    for attack_label, n_target in target_counts.items():
        print(f"\nTraining class-specific DAE for: {attack_label}")

        df_class = df[df[label_col] == attack_label].copy()

        if len(df_class) < 10:
            raise ValueError(f"Not enough rows for class '{attack_label}' to train a stable autoencoder.")

        X = df_class[feature_cols].values.astype("float32")

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        X_train, X_val = train_test_split(
            X_scaled,
            test_size=0.2,
            random_state=SEED,
            shuffle=True
        )

        autoencoder = build_dae(
            input_dim=len(feature_cols),
            hidden_layers=hidden_layers,
            latent_dim=latent_dim,
            corruption_rate=corruption_rate,
            activation_fn=activation_fn
        )

        history = autoencoder.fit(
            X_train,
            X_train,
            validation_data=(X_val, X_val),
            epochs=epochs,
            batch_size=batch_size,
            shuffle=True,
            callbacks=[early_stopping],
            verbose=1
        )

        # Store training summary for later comparison
        metrics_rows.append({
            "variant": variant_name,
            "attack_label": attack_label,
            "epochs_trained": len(history.history["loss"]),
            "final_train_loss": float(history.history["loss"][-1]),
            "final_val_loss": float(history.history["val_loss"][-1]),
            "best_val_loss": float(np.min(history.history["val_loss"]))
        })

        # Sample real rows with replacement to preserve class and state structure
        sample_idx = np.random.choice(len(df_class), size=n_target, replace=True)
        sampled_rows = df_class.iloc[sample_idx].reset_index(drop=True)

        # Transform the sampled numeric features into scaled space
        X_base = scaler.transform(sampled_rows[feature_cols].values.astype("float32"))

        # Add Gaussian noise before reconstruction
        noise = np.random.normal(loc=0.0, scale=corruption_rate, size=X_base.shape)
        X_noisy = X_base + noise

        # Reconstruct noisy inputs with the trained autoencoder
        X_reconstructed = autoencoder.predict(X_noisy, batch_size=batch_size, verbose=0)

        # Transform reconstructed values back to original scale
        X_synth = scaler.inverse_transform(X_reconstructed)

        # Create synthetic rows
        synth_df = pd.DataFrame(X_synth, columns=feature_cols)
        synth_df[state_col] = sampled_rows[state_col].values
        synth_df[label_col] = attack_label

        # Clip values per class to reduce unrealistic outliers
        for col in feature_cols:
            lower = df_class[col].quantile(0.001)
            upper = df_class[col].quantile(0.999)
            synth_df[col] = synth_df[col].clip(lower=lower, upper=upper)

        synthetic_parts.append(synth_df)

    # Merge all synthetic class parts into one dataset
    df_synth = pd.concat(synthetic_parts, ignore_index=True)
    df_synth = df_synth[feature_cols + [state_col, label_col]]

    # Save synthetic data for this variant
    output_file = os.path.join(output_dir, f"synthetic_dataset_dae_{variant_name}.csv")
    df_synth.to_csv(output_file, index=False)

    # Save training metrics for this variant
    metrics_df = pd.DataFrame(metrics_rows)
    metrics_file = os.path.join(output_dir, f"synthetic_dataset_dae_{variant_name}_metrics.csv")
    metrics_df.to_csv(metrics_file, index=False)

    print(f"\nSaved synthetic dataset to: {output_file}")
    print(f"Saved metrics to: {metrics_file}")
    print(df_synth[label_col].value_counts())
    print(df_synth.head())

    return df_synth, metrics_df


# --------------------------------------------------
# Run all variants
# --------------------------------------------------
all_metrics = []

for variant_name, config in variants.items():
    _, metrics_df = generate_synthetic_dataset(df, variant_name, config)
    all_metrics.append(metrics_df)

# Combine all metrics into one file for direct comparison
all_metrics_df = pd.concat(all_metrics, ignore_index=True)
all_metrics_output = os.path.join(output_dir, "synthetic_dataset_dae_all_variants_metrics.csv")
all_metrics_df.to_csv(all_metrics_output, index=False)

print(f"\nSaved combined metrics to: {all_metrics_output}")
print(all_metrics_df)

Target counts: {'none': 10053, 'Backdoor': 14796, 'syn-flood': 9463}


KeyError: 'corruption_rate'

In [7]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import AdamW

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

input_path = "../datasets/CICEVSE2024/synthetic_extension/train_dataset.csv"
output_dir = "../datasets/CICEVSE2024/synthetic_extension/"
os.makedirs(output_dir, exist_ok=True)

feature_cols = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]
state_col = "State"
label_col = "Attack"

target_total_rows = 34312
target_distribution = {
    "none": 0.2930,
    "Backdoor": 0.4312,
    "syn-flood": 0.2758
}

variants = {
    "best_ae_variant": {
        "input_dim": 4,
        "latent_dim": 8,
        "hidden_layers": [256, 128],
        "dropout": 0.0,
        "epochs": 49,
        "batch_size": 64,
        "learning_rate": 5e-05,
        "weight_decay": 0.0
    },
    "variant_a": {
        "input_dim": 4,
        "latent_dim": 8,
        "hidden_layers": [64, 32],
        "dropout": 0.0,
        "epochs": 47,
        "batch_size": 128,
        "learning_rate": 0.001,
        "weight_decay": 0.0
    },
    "variant_b": {
        "input_dim": 4,
        "latent_dim": 8,
        "hidden_layers": [256, 128],
        "dropout": 0.0,
        "epochs": 38,
        "batch_size": 64,
        "learning_rate": 5e-05,
        "weight_decay": 1e-05
    },
    "variant_c": {
        "input_dim": 4,
        "latent_dim": 8,
        "hidden_layers": [256, 128],
        "dropout": 0.0,
        "epochs": 45,
        "batch_size": 128,
        "learning_rate": 1e-04,
        "weight_decay": 0.0
    }
}

df = pd.read_csv(input_path)

required_cols = feature_cols + [state_col, label_col]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].copy()
df = df[df[label_col].isin(target_distribution.keys())].dropna().reset_index(drop=True)

if df.empty:
    raise ValueError("No valid rows found after filtering for required attack labels.")

target_counts = {k: int(target_total_rows * v) for k, v in target_distribution.items()}
remainder = target_total_rows - sum(target_counts.values())
if remainder != 0:
    largest_class = max(target_distribution, key=target_distribution.get)
    target_counts[largest_class] += remainder

print("Target counts:", target_counts)

def build_ae(input_dim, hidden_layers, latent_dim, dropout, learning_rate, weight_decay):
    inp = Input(shape=(input_dim,), name="input")
    x = inp

    for i, units in enumerate(hidden_layers):
        x = Dense(units, activation="relu", name=f"encoder_dense_{i+1}")(x)
        if dropout > 0:
            x = Dropout(dropout, name=f"dropout_{i+1}")(x)

    latent = Dense(latent_dim, activation="relu", name="latent")(x)
    x = latent

    for i, units in enumerate(reversed(hidden_layers)):
        x = Dense(units, activation="relu", name=f"decoder_dense_{i+1}")(x)
        if dropout > 0:
            x = Dropout(dropout, name=f"decoder_dropout_{i+1}")(x)

    out = Dense(input_dim, activation="linear", name="reconstruction")(x)

    model = Model(inputs=inp, outputs=out, name="autoencoder")
    optimizer = AdamW(learning_rate=learning_rate, weight_decay=weight_decay)
    model.compile(optimizer=optimizer, loss="mse")
    return model

def generate_synthetic_dataset(df, variant_name, config):
    hidden_layers = config["hidden_layers"]
    latent_dim = config["latent_dim"]
    dropout = config["dropout"]
    epochs = config["epochs"]
    batch_size = config["batch_size"]
    learning_rate = config["learning_rate"]
    weight_decay = config["weight_decay"]

    print(f"\n{'=' * 60}")
    print(f"Running variant: {variant_name}")
    print(f"Configuration: {config}")
    print(f"{'=' * 60}")

    early_stopping = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=10,
        min_delta=1e-4,
        restore_best_weights=True,
        start_from_epoch=10,
        verbose=1
    )

    synthetic_parts = []
    metrics_rows = []

    for attack_label, n_target in target_counts.items():
        print(f"\nTraining class-specific AE for: {attack_label}")

        df_class = df[df[label_col] == attack_label].copy()
        if len(df_class) < 10:
            raise ValueError(f"Not enough rows for class '{attack_label}' to train a stable autoencoder.")

        X = df_class[feature_cols].values.astype("float32")
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        X_train, X_val = train_test_split(
            X_scaled, test_size=0.2, random_state=SEED, shuffle=True
        )

        autoencoder = build_ae(
            input_dim=len(feature_cols),
            hidden_layers=hidden_layers,
            latent_dim=latent_dim,
            dropout=dropout,
            learning_rate=learning_rate,
            weight_decay=weight_decay
        )

        history = autoencoder.fit(
            X_train, X_train,
            validation_data=(X_val, X_val),
            epochs=epochs,
            batch_size=batch_size,
            shuffle=True,
            callbacks=[early_stopping],
            verbose=1
        )

        metrics_rows.append({
            "variant": variant_name,
            "attack_label": attack_label,
            "epochs_trained": len(history.history["loss"]),
            "final_train_loss": float(history.history["loss"][-1]),
            "final_val_loss": float(history.history["val_loss"][-1]),
            "best_val_loss": float(np.min(history.history["val_loss"]))
        })

        sample_idx = np.random.choice(len(df_class), size=n_target, replace=True)
        sampled_rows = df_class.iloc[sample_idx].reset_index(drop=True)

        X_base = scaler.transform(sampled_rows[feature_cols].values.astype("float32"))
        X_reconstructed = autoencoder.predict(X_base, batch_size=batch_size, verbose=0)
        X_synth = scaler.inverse_transform(X_reconstructed)

        synth_df = pd.DataFrame(X_synth, columns=feature_cols)
        synth_df[state_col] = sampled_rows[state_col].values
        synth_df[label_col] = attack_label

        for col in feature_cols:
            lower = df_class[col].quantile(0.001)
            upper = df_class[col].quantile(0.999)
            synth_df[col] = synth_df[col].clip(lower=lower, upper=upper)

        synthetic_parts.append(synth_df)

    df_synth = pd.concat(synthetic_parts, ignore_index=True)
    df_synth = df_synth[feature_cols + [state_col, label_col]]

    output_file = os.path.join(output_dir, f"synthetic_dataset_ae_{variant_name}.csv")
    df_synth.to_csv(output_file, index=False)

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_file = os.path.join(output_dir, f"synthetic_dataset_ae_{variant_name}_metrics.csv")
    metrics_df.to_csv(metrics_file, index=False)

    print(f"\nSaved synthetic dataset to: {output_file}")
    print(f"Saved metrics to: {metrics_file}")

    return df_synth, metrics_df

all_metrics = []
for variant_name, config in variants.items():
    _, metrics_df = generate_synthetic_dataset(df, variant_name, config)
    all_metrics.append(metrics_df)

all_metrics_df = pd.concat(all_metrics, ignore_index=True)
all_metrics_output = os.path.join(output_dir, "synthetic_dataset_ae_all_variants_metrics.csv")
all_metrics_df.to_csv(all_metrics_output, index=False)

print(f"\nSaved combined metrics to: {all_metrics_output}")
print(all_metrics_df)

Target counts: {'none': 10053, 'Backdoor': 14796, 'syn-flood': 9463}

Running variant: best_ae_variant
Configuration: {'input_dim': 4, 'latent_dim': 8, 'hidden_layers': [256, 128], 'dropout': 0.0, 'epochs': 49, 'batch_size': 64, 'learning_rate': 5e-05, 'weight_decay': 0.0}

Training class-specific AE for: none
Epoch 1/49


E0000 00:00:1784620786.178738   64832 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.8253 - val_loss: 0.4952
Epoch 2/49
126/126 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2900 - val_loss: 0.1898
Epoch 3/49
126/126 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1360 - val_loss: 0.0935
Epoch 4/49
126/126 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0665 - val_loss: 0.0483
Epoch 5/49
126/126 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0272 - val_loss: 0.0111
Epoch 6/49
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0057 - val_loss: 0.0036
Epoch 7/49
126/126 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0026 - val_loss: 0.0020
Epoch 8/49
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0016 - val_loss: 0.0013
Epoch 9/49
126/126 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0011 - val_loss: 9.7465e-04
Epoch 10/49
126/126 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 8.4657e-04 - val_loss: 7.5760e-04
Epoch 11/49
126/126 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 6.7239e-04 - val_loss: 6.0617e-04
Epoch 12/49
126/126 ━━━━━━━━━━━━━━━━━━━━ 1s

185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 5.5704e-05 - val_loss: 6.1393e-05
Epoch 31: early stopping
Restoring model weights from the end of the best epoch: 21.

Training class-specific AE for: syn-flood
Epoch 1/49
119/119 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.8781 - val_loss: 0.5719
Epoch 2/49
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2392 - val_loss: 0.1102
Epoch 3/49
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0861 - val_loss: 0.0768
Epoch 4/49
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0567 - val_loss: 0.0498
Epoch 5/49
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0370 - val_loss: 0.0342
Epoch 6/49
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0244 - val_loss: 0.0201
Epoch 7/49
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0118 - val_loss: 0.0088
Epoch 8/49
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0053 - val_loss: 0.0043
Epoch 9/49
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0026 - val_loss: 0.0022
Epoch 10/49
119

Epoch 17/47
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.6089e-04 - val_loss: 2.4576e-04
Epoch 18/47
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.2352e-04 - val_loss: 2.2391e-04
Epoch 19/47
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2.0332e-04 - val_loss: 2.0684e-04
Epoch 20/47
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1.9095e-04 - val_loss: 1.9670e-04
Epoch 21/47
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.7940e-04 - val_loss: 1.8033e-04
Epoch 22/47
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.7929e-04 - val_loss: 1.7375e-04
Epoch 23/47
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.8304e-04 - val_loss: 1.5951e-04
Epoch 24/47
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.7134e-04 - val_loss: 1.6326e-04
Epoch 25/47
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.7112e-04 - val_loss: 1.6351e-04
Epoch 26/47
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.7000e-04 - val_loss: 1.6149e-04
Epoch 27/47
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.6111e-04 

60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.9515e-04 - val_loss: 5.3363e-04
Epoch 18/47
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.7507e-04 - val_loss: 5.1632e-04
Epoch 19/47
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.5811e-04 - val_loss: 4.7958e-04
Epoch 20/47
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.3273e-04 - val_loss: 4.5401e-04
Epoch 21/47
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.1363e-04 - val_loss: 4.7768e-04
Epoch 22/47
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.0199e-04 - val_loss: 5.2780e-04
Epoch 23/47
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.9434e-04 - val_loss: 5.5241e-04
Epoch 24/47
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7973e-04 - val_loss: 5.7038e-04
Epoch 25/47
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.6748e-04 - val_loss: 5.5198e-04
Epoch 26/47
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.5388e-04 - val_loss: 5.4311e-04
Epoch 27/47
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.4148e-04 - val_loss: 

185/185 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 6.6226e-04 - val_loss: 7.1376e-04
Epoch 12/38
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 5.3454e-04 - val_loss: 5.9209e-04
Epoch 13/38
185/185 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.4317e-04 - val_loss: 5.0234e-04
Epoch 14/38
185/185 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.7525e-04 - val_loss: 4.3238e-04
Epoch 15/38
185/185 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.2184e-04 - val_loss: 3.7498e-04
Epoch 16/38
185/185 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.7866e-04 - val_loss: 3.2830e-04
Epoch 17/38
185/185 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.4356e-04 - val_loss: 2.9053e-04
Epoch 18/38
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.1517e-04 - val_loss: 2.6117e-04
Epoch 19/38
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.9191e-04 - val_loss: 2.3743e-04
Epoch 20/38
185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.7255e-04 - val_loss: 2.1484e-04
Epoch 21/38
185/185 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1

63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 0.8686 - val_loss: 0.5821
Epoch 2/45
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.3106 - val_loss: 0.1840
Epoch 3/45
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.1273 - val_loss: 0.0756
Epoch 4/45
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0447 - val_loss: 0.0237
Epoch 5/45
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 0.0144 - val_loss: 0.0092
Epoch 6/45
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0062 - val_loss: 0.0045
Epoch 7/45
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.0032 - val_loss: 0.0026
Epoch 8/45
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 0.0019 - val_loss: 0.0015
Epoch 9/45
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0012 - val_loss: 9.4506e-04
Epoch 10/45
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 8.1724e-04 - val_loss: 6.8544e-04
Epoch 11/45
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - loss: 6.3226e-04 - val_loss: 5.4603e-04
Epoch 12/45
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - 

60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.0015 - val_loss: 0.0015
Epoch 8/45
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0011 - val_loss: 0.0011
Epoch 9/45
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 8.6625e-04 - val_loss: 9.1939e-04
Epoch 10/45
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 7.0403e-04 - val_loss: 7.6185e-04
Epoch 11/45
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 5.8400e-04 - val_loss: 6.4276e-04
Epoch 12/45
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 4.9233e-04 - val_loss: 5.4825e-04
Epoch 13/45
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 4.1994e-04 - val_loss: 4.7412e-04
Epoch 14/45
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.6110e-04 - val_loss: 4.1480e-04
Epoch 15/45
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.1339e-04 - val_loss: 3.6642e-04
Epoch 16/45
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7473e-04 - val_loss: 3.2668e-04
Epoch 17/45
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.4319e-04 - val_loss: 2.9203e-04
Epoc

In [ ]:
# könnte training auch auf syn + echte daten

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense, GaussianNoise
from tensorflow.keras.callbacks import EarlyStopping


# --------------------------------------------------
# Reproducibility
# --------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# --------------------------------------------------
# Paths
# --------------------------------------------------
input_path = "../datasets/CICEVSE2024/synthetic_extension/train_dataset.csv"
output_dir = "../datasets/CICEVSE2024/synthetic_extension/"
os.makedirs(output_dir, exist_ok=True)


# --------------------------------------------------
# Dataset configuration
# --------------------------------------------------
feature_cols = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]
state_col = "State"
label_col = "Attack"

target_total_rows = 34312
target_distribution = {
    "none": 0.2930,
    "Backdoor": 0.4312,
    "syn-flood": 0.2758
}


# --------------------------------------------------
# Model variants
# baseline_variant: current baseline model
# variant_a: smaller model with smaller latent space
# variant_b: stronger noise for denoising robustness
# variant_c: wider model with larger latent space
# --------------------------------------------------
variants = {
    "baseline_variant": {
        "corruption_rate": 0.2,
        "hidden_layers": [64, 32, 16, 8, 4, 4, 8, 16, 32, 64],
        "latent_dim": 4,
        "activation_fn": "relu",
        "epochs": 100,
        "batch_size": 32
    },
    "variant_a": {
        "corruption_rate": 0.1,
        "hidden_layers": [32, 16, 8, 4, 4, 8, 16, 32],
        "latent_dim": 2,
        "activation_fn": "relu",
        "epochs": 100,
        "batch_size": 32
    },
    "variant_b": {
        "corruption_rate": 0.3,
        "hidden_layers": [64, 32, 16, 8, 8, 16, 32, 64],
        "latent_dim": 4,
        "activation_fn": "relu",
        "epochs": 100,
        "batch_size": 32
    },
    "variant_c": {
        "corruption_rate": 0.2,
        "hidden_layers": [128, 64, 32, 16, 8, 8, 16, 32, 64, 128],
        "latent_dim": 8,
        "activation_fn": "relu",
        "epochs": 100,
        "batch_size": 32
    }
}


# --------------------------------------------------
# Load and validate dataset
# --------------------------------------------------
df = pd.read_csv(input_path)

required_cols = feature_cols + [state_col, label_col]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].copy()
df = df[df[label_col].isin(target_distribution.keys())].dropna().reset_index(drop=True)

if df.empty:
    raise ValueError("No valid rows found after filtering for required attack labels.")


# --------------------------------------------------
# Encode State as a single binary feature
# Adjust the mapping here if your dataset uses other labels
# --------------------------------------------------
state_map = {
    "idle": 0,
    "charging": 1
}

df[state_col] = df[state_col].astype(str).str.strip().str.lower().map(state_map)

if df[state_col].isna().any():
    unknown_states = df.loc[df[state_col].isna(), state_col].unique()
    raise ValueError(f"Unknown values found in State column: {unknown_states}")


# --------------------------------------------------
# Compute target row count per attack label
# --------------------------------------------------
target_counts = {k: int(target_total_rows * v) for k, v in target_distribution.items()}
remainder = target_total_rows - sum(target_counts.values())

if remainder != 0:
    largest_class = max(target_distribution, key=target_distribution.get)
    target_counts[largest_class] += remainder

print("Target counts:", target_counts)


# --------------------------------------------------
# Build denoising autoencoder
# The encoder uses the first half of the hidden layers.
# The decoder uses the second half of the hidden layers.
# --------------------------------------------------
def build_dae(input_dim, hidden_layers, latent_dim, corruption_rate, activation_fn="relu"):
    inp = Input(shape=(input_dim,), name="clean_input")
    x = GaussianNoise(stddev=corruption_rate, name="input_noise")(inp)

    split_index = len(hidden_layers) // 2
    encoder_layers = hidden_layers[:split_index]
    decoder_layers = hidden_layers[split_index:]

    for i, units in enumerate(encoder_layers):
        x = Dense(units, activation=activation_fn, name=f"encoder_dense_{i+1}")(x)

    latent = Dense(latent_dim, activation=activation_fn, name="latent")(x)

    x = latent
    for i, units in enumerate(decoder_layers):
        x = Dense(units, activation=activation_fn, name=f"decoder_dense_{i+1}")(x)

    out = Dense(input_dim, activation="linear", name="reconstruction")(x)

    autoencoder = Model(inputs=inp, outputs=out, name="denoising_autoencoder")
    autoencoder.compile(optimizer="adam", loss="mse")
    return autoencoder


# --------------------------------------------------
# Train and generate synthetic data for one variant
# State is included directly as a single binary feature
# during both training and synthesis
# --------------------------------------------------
def generate_synthetic_dataset(df, variant_name, config):
    corruption_rate = config["corruption_rate"]
    hidden_layers = config["hidden_layers"]
    latent_dim = config["latent_dim"]
    activation_fn = config["activation_fn"]
    epochs = config["epochs"]
    batch_size = config["batch_size"]

    print(f"\n{'=' * 60}")
    print(f"Running variant: {variant_name}")
    print(f"Configuration: {config}")
    print(f"{'=' * 60}")

    early_stopping = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=10,
        min_delta=1e-4,
        restore_best_weights=True,
        start_from_epoch=10,
        verbose=1
    )

    synthetic_parts = []
    metrics_rows = []

    for attack_label, n_target in target_counts.items():
        print(f"\nTraining class-specific DAE for: {attack_label}")

        df_class = df[df[label_col] == attack_label].copy()

        if len(df_class) < 10:
            raise ValueError(f"Not enough rows for class '{attack_label}' to train a stable autoencoder.")

        # Scale only the numeric feature columns
        scaler = StandardScaler()
        X_num = scaler.fit_transform(df_class[feature_cols].values.astype("float32"))

        # Keep State as a single binary column
        X_state = df_class[[state_col]].values.astype("float32")

        # Combine numeric features and binary State into one input matrix
        X_full = np.hstack([X_num, X_state]).astype("float32")

        X_train, X_val = train_test_split(
            X_full,
            test_size=0.2,
            random_state=SEED,
            shuffle=True
        )

        autoencoder = build_dae(
            input_dim=X_full.shape[1],
            hidden_layers=hidden_layers,
            latent_dim=latent_dim,
            corruption_rate=corruption_rate,
            activation_fn=activation_fn
        )

        history = autoencoder.fit(
            X_train,
            X_train,
            validation_data=(X_val, X_val),
            epochs=epochs,
            batch_size=batch_size,
            shuffle=True,
            callbacks=[early_stopping],
            verbose=1
        )

        # Store training summary for later comparison
        metrics_rows.append({
            "variant": variant_name,
            "attack_label": attack_label,
            "epochs_trained": len(history.history["loss"]),
            "final_train_loss": float(history.history["loss"][-1]),
            "final_val_loss": float(history.history["val_loss"][-1]),
            "best_val_loss": float(np.min(history.history["val_loss"]))
        })

        # Sample real rows with replacement to preserve attack-specific structure
        sample_idx = np.random.choice(len(df_class), size=n_target, replace=True)
        sampled_rows = df_class.iloc[sample_idx].reset_index(drop=True)

        # Transform numeric features and keep binary State
        X_num_base = scaler.transform(sampled_rows[feature_cols].values.astype("float32"))
        X_state_base = sampled_rows[[state_col]].values.astype("float32")
        X_base = np.hstack([X_num_base, X_state_base]).astype("float32")

        # Add Gaussian noise before reconstruction
        noise = np.random.normal(loc=0.0, scale=corruption_rate, size=X_base.shape)
        X_noisy = X_base + noise

        # Reconstruct noisy inputs with the trained autoencoder
        X_reconstructed = autoencoder.predict(X_noisy, batch_size=batch_size, verbose=0)

        # Split reconstructed values into numeric and binary State parts
        num_dim = len(feature_cols)
        X_num_reconstructed = X_reconstructed[:, :num_dim]
        X_state_reconstructed = X_reconstructed[:, num_dim:]

        # Transform numeric values back to original scale
        X_synth_num = scaler.inverse_transform(X_num_reconstructed)

        # Convert reconstructed State back to binary values
        X_synth_state = (X_state_reconstructed >= 0.5).astype(int).ravel()

        # Create synthetic rows
        synth_df = pd.DataFrame(X_synth_num, columns=feature_cols)
        synth_df[state_col] = X_synth_state
        synth_df[label_col] = attack_label

        # Clip numeric values per class to reduce unrealistic outliers
        for col in feature_cols:
            lower = df_class[col].quantile(0.001)
            upper = df_class[col].quantile(0.999)
            synth_df[col] = synth_df[col].clip(lower=lower, upper=upper)

        synthetic_parts.append(synth_df)

    # Merge all synthetic class parts into one dataset
    df_synth = pd.concat(synthetic_parts, ignore_index=True)
    df_synth = df_synth[feature_cols + [state_col, label_col]]

    # Save synthetic data for this variant
    output_file = os.path.join(output_dir, f"synthetic_dataset_dae_{variant_name}.csv")
    df_synth.to_csv(output_file, index=False)

    # Save training metrics for this variant
    metrics_df = pd.DataFrame(metrics_rows)
    metrics_file = os.path.join(output_dir, f"synthetic_dataset_dae_{variant_name}_metrics.csv")
    metrics_df.to_csv(metrics_file, index=False)

    print(f"\nSaved synthetic dataset to: {output_file}")
    print(f"Saved metrics to: {metrics_file}")
    print(df_synth[label_col].value_counts())
    print(df_synth.head())

    return df_synth, metrics_df


# --------------------------------------------------
# Run all variants
# --------------------------------------------------
all_metrics = []

for variant_name, config in variants.items():
    _, metrics_df = generate_synthetic_dataset(df, variant_name, config)
    all_metrics.append(metrics_df)

# Combine all metrics into one file for direct comparison
all_metrics_df = pd.concat(all_metrics, ignore_index=True)
all_metrics_output = os.path.join(output_dir, "synthetic_dataset_dae_all_variants_metrics.csv")
all_metrics_df.to_csv(all_metrics_output, index=False)

print(f"\nSaved combined metrics to: {all_metrics_output}")
print(all_metrics_df)